# Tutorial 06: RingTransaction — IICA-Compliant HLLSet Construction

This tutorial introduces the `RingTransaction` module, which provides an ephemeral workspace for building HLLSets while enforcing the IICA principles.

## What You'll Learn

1. **Transaction Lifecycle** — begin → ingest → commit/rollback
2. **IICA Enforcement** — How transactions guarantee algebraic closure
3. **Token Ingestion** — Building HLLSets through the ring algebra
4. **Basis Inspection** — Understanding Gaussian elimination results
5. **Search & Filter** — Finding relevant basis vectors
6. **Commit Results** — The frozen, immutable output

## The IICA Principles

| Principle | Transaction Enforcement |
|-----------|------------------------|
| **I** — Immutability | Committed results are frozen; mid-transaction state is ephemeral |
| **I** — Idempotence | Re-ingesting same tokens → same bitvector → same SHA1 |
| **C** — Composability | Every stage consumes and produces HLLSets |
| **A** — Algebraic Closure | Ring algebra provides lossless compression and basis extraction |

In [1]:
# Import modules
import sys
sys.path.insert(0, '..')

from core.ring_transaction import (
    RingTransaction,
    TransactionResult,
    TransactionPhase,
    IngestRecord,
    RingDelta,
)
from core.hllset import HLLSet
from core.bitvector_ring import BitVectorRing

## 1. Transaction Lifecycle

A transaction follows a strict lifecycle:

```
┌─ ACTIVE ──────────────────────────────────┐
│  ingest()  →  ingest()  →  search()       │
└─────────────────┬─────────────────────────┘
                  │
        ┌─────────┴─────────┐
        ▼                   ▼
  COMMITTED           ROLLED_BACK
  (result)            (discarded)
```

Once committed or rolled back, the transaction cannot be modified.

In [2]:
# Create a fresh transaction
tx = RingTransaction(p_bits=10)

print(f"Transaction created")
print(f"  Phase: {tx._phase.value}")
print(f"  Initial basis count: {tx.basis_count}")

Transaction created
  Phase: active
  Initial basis count: 0


In [3]:
# Ingest some tokens
record1 = tx.ingest(["machine", "learning", "deep", "neural"])

print(f"Ingest record: {record1}")
print(f"  Batch ID: {record1.batch_id[:16]}...")
print(f"  Vector ID: {record1.vector_id}")
print(f"  Tokens: {record1.tokens}")
print(f"  Popcount: {record1.popcount}")
print(f"  Rank delta: {record1.rank_delta}")

Ingest record: IngestRecord(745478dc…, 4 tokens, pop=4, Δrank=+1)
  Batch ID: 745478dcd6e251ba...
  Vector ID: 0
  Tokens: ('machine', 'learning', 'deep', 'neural')
  Popcount: 4
  Rank delta: 1


In [4]:
# Ingest more batches
record2 = tx.ingest(["natural", "language", "processing"], label="nlp")
record3 = tx.ingest(["computer", "vision", "algorithms"], label="cv")

print(f"After 3 ingests:")
print(f"  Basis count: {tx.basis_count}")
print(f"  Record 2: {record2}")
print(f"  Record 3: {record3}")

After 3 ingests:
  Basis count: 3
  Record 2: IngestRecord(6610fe37…, 3 tokens, pop=3, Δrank=+1)
  Record 3: IngestRecord(95aa2f93…, 3 tokens, pop=3, Δrank=+1)


In [5]:
# Commit the transaction
result = tx.commit()

print(f"Transaction committed!")
print(f"  Phase: {tx._phase.value}")
print(f"  Result: {result}")

Transaction committed!
  Phase: committed
  Result: TransactionResult(90beb644…, |M|≈12, RingDelta(rank 0→3, +3 bases, 10 tokens in 3 batches))


In [6]:
# Cannot modify after commit
try:
    tx.ingest(["more", "tokens"])
except RuntimeError as e:
    print(f"Expected error: {e}")

Expected error: Transaction is committed; cannot modify.


## 2. Context Manager Pattern

Transactions support the context manager protocol for automatic rollback on exceptions.

In [7]:
# Using context manager — auto-rollback on exception
with RingTransaction(p_bits=10) as tx:
    tx.ingest(["alpha", "beta", "gamma"])
    tx.ingest(["delta", "epsilon"])
    result = tx.commit()

print(f"Context manager completed: {result}")

Context manager completed: TransactionResult(7245721d…, |M|≈6, RingDelta(rank 0→2, +2 bases, 5 tokens in 2 batches))


In [8]:
# Automatic rollback on exception
try:
    with RingTransaction(p_bits=10) as tx:
        tx.ingest(["some", "tokens"])
        raise ValueError("Simulated failure")
        # commit() never reached
except ValueError:
    print(f"Exception handled, transaction auto-rolled back")
    print(f"Phase: {tx._phase.value}")

Exception handled, transaction auto-rolled back
Phase: rolled_back


## 3. Understanding Ingest Records

Each `ingest()` call returns an `IngestRecord` — a frozen provenance record:

- `batch_id`: SHA1(sorted tokens) — content address
- `vector_id`: Ring vector ID from Gaussian elimination
- `tokens`: Original tokens (frozen tuple)
- `reg_zeros`: Per-token (register, zeros) pairs
- `rank_delta`: How much this batch increased the ring rank

In [9]:
with RingTransaction(p_bits=10) as tx:
    # First batch — likely extends the basis
    r1 = tx.ingest(["machine", "learning", "neural", "networks"])
    print(f"Batch 1: Δrank = {r1.rank_delta:+d}, extending = {r1.is_basis_extending}")
    
    # Second batch — may or may not extend
    r2 = tx.ingest(["deep", "learning", "models"])
    print(f"Batch 2: Δrank = {r2.rank_delta:+d}, extending = {r2.is_basis_extending}")
    
    # Duplicate batch — should NOT extend (idempotent)
    r3 = tx.ingest(["machine", "learning", "neural", "networks"])  # Same as r1
    print(f"Batch 3 (duplicate): Δrank = {r3.rank_delta:+d}, extending = {r3.is_basis_extending}")
    
    result = tx.commit()

print(f"\nTotal basis vectors: {len(result.basis_vectors)}")

Batch 1: Δrank = +1, extending = True
Batch 2: Δrank = +1, extending = True
Batch 3 (duplicate): Δrank = +0, extending = False

Total basis vectors: 2


## 4. Ring Delta — Compression Statistics

The `RingDelta` shows what changed during the transaction:

- **rank_before / rank_after**: Basis rank change
- **compression_ratio**: tokens / rank (higher = better compression)
- **independent_count**: Batches that added new basis vectors
- **dependent_count**: Batches fully in existing span

In [10]:
with RingTransaction(p_bits=10) as tx:
    tx.ingest(["alpha", "beta", "gamma", "delta"])
    tx.ingest(["epsilon", "zeta", "eta", "theta"])
    tx.ingest(["iota", "kappa", "lambda", "mu"])
    result = tx.commit()

delta = result.ring_delta
print(f"Ring Delta:")
print(f"  Rank: {delta.rank_before} → {delta.rank_after}")
print(f"  New basis vectors: {delta.new_basis_count}")
print(f"  Total tokens: {delta.total_tokens}")
print(f"  Compression ratio: {delta.compression_ratio:.2f}x")
print(f"  Independent batches: {delta.independent_count}")
print(f"  Dependent batches: {delta.dependent_count}")

Ring Delta:
  Rank: 0 → 3
  New basis vectors: 3
  Total tokens: 12
  Compression ratio: 4.00x
  Independent batches: 3
  Dependent batches: 0


## 5. Basis Inspection

Before committing, you can inspect the basis vectors built during ingestion.

In [15]:
with RingTransaction(p_bits=10) as tx:
    tx.ingest(["machine", "learning", "deep"])
    tx.ingest(["natural", "language", "processing"])
    
    # Inspect basis before commit
    print(f"Basis vectors in draft ring:")
    for idx, bv in tx.get_basis_vectors():
        print(f"  [{idx}] popcount={bv.popcount()}")
    
    result = tx.commit()

Basis vectors in draft ring:
  [0] popcount=3
  [1] popcount=3


In [16]:
# Get detailed basis info
with RingTransaction(p_bits=10) as tx:
    tx.ingest(["the", "quick", "brown", "fox", "jumps"])
    
    info = tx.get_basis_info()
    print(f"Basis info:")
    for item in info:
        print(f"  Index {item['index']}: leading_bit={item['leading_bit']}, popcount={item['popcount']}")
    
    result = tx.commit()

Basis info:
  Index 0: leading_bit=30721, popcount=5


## 6. TransactionResult — The Frozen Output

After `commit()`, you receive a `TransactionResult` — completely immutable:

- `merged_hllset`: The primary HLLSet (union of selected vectors)
- `basis_vectors`: Frozen tuple of BitVectors
- `token_lut`: TokenLUT built during ingestion
- `ingest_records`: Full provenance chain

In [17]:
with RingTransaction(p_bits=10) as tx:
    tx.ingest(["machine", "learning", "deep", "neural", "networks"])
    tx.ingest(["natural", "language", "processing", "nlp"])
    result = tx.commit()

print(f"TransactionResult:")
print(f"  Transaction ID: {result.transaction_id[:16]}...")
print(f"  Timestamp: {result.timestamp:.2f}")
print(f"  Merged HLLSet cardinality: ~{result.merged_hllset.cardinality():.0f}")
print(f"  Basis vectors: {len(result.basis_vectors)}")
print(f"  Ingest records: {len(result.ingest_records)}")

TransactionResult:
  Transaction ID: 7f1b69491c91ada1...
  Timestamp: 1775938275.92
  Merged HLLSet cardinality: ~10
  Basis vectors: 2
  Ingest records: 2


In [18]:
# The merged HLLSet is a regular HLLSet — all operations work
merged = result.merged_hllset
print(f"Merged HLLSet:")
print(f"  Type: {type(merged).__name__}")
print(f"  Name (SHA1): {merged.name[:16]}...")
print(f"  Cardinality: ~{merged.cardinality():.0f}")

Merged HLLSet:
  Type: HLLSet
  Name (SHA1): 7f1b69491c91ada1...
  Cardinality: ~10


In [19]:
# Provenance through ingest records
print(f"Ingest provenance:")
for i, rec in enumerate(result.ingest_records):
    print(f"  [{i}] {rec.label or '(no label)'}: {len(rec.tokens)} tokens, Δrank={rec.rank_delta:+d}")

Ingest provenance:
  [0] (no label): 5 tokens, Δrank=+1
  [1] (no label): 4 tokens, Δrank=+1


## 7. Branching from an Existing Ring

Transactions can branch from an existing `BitVectorRing`, creating a deep copy for isolation.

In [ ]:
# Create a base ring with some content
base_ring = BitVectorRing(N=1024 * 32)  # matches p_bits=10

# Build initial content through a transaction
with RingTransaction(base_ring=base_ring, p_bits=10) as tx:
    tx.ingest(["foundation", "content", "base", "layer"])
    result1 = tx.commit()

print(f"Base transaction: {result1.ring_delta}")

In [ ]:
# Branch a new transaction from the base
# NOTE: base_ring is deep-copied, so original is never mutated
with RingTransaction(base_ring=base_ring, p_bits=10) as tx:
    tx.ingest(["additional", "content", "on", "top"])
    result2 = tx.commit()

print(f"Branched transaction: {result2.ring_delta}")
print(f"Original base ring unchanged: rank = {base_ring.rank()}")

## 8. Batch Ingestion

For multiple batches, use `ingest_batches()` for convenience.

In [ ]:
batches = [
    ["machine", "learning"],
    ["deep", "neural", "networks"],
    ["natural", "language", "processing"],
    ["computer", "vision"],
]

labels = ["ml", "dnn", "nlp", "cv"]

with RingTransaction(p_bits=10) as tx:
    records = tx.ingest_batches(batches, labels=labels)
    
    print(f"Batch ingestion results:")
    for rec in records:
        print(f"  {rec.label}: {len(rec.tokens)} tokens, Δrank={rec.rank_delta:+d}")
    
    result = tx.commit()

print(f"\nFinal: {result.ring_delta}")

## 9. Search — Finding Relevant Basis Vectors

The `search()` method finds basis vectors and ingested vectors that overlap with a query HLLSet or BitVector.

In [ ]:
with RingTransaction(p_bits=10) as tx:
    # Ingest diverse content
    tx.ingest(["machine", "learning", "algorithms"])
    tx.ingest(["deep", "neural", "networks"])
    tx.ingest(["natural", "language", "processing"])
    
    # Create a query HLLSet
    query = HLLSet.from_batch(["machine", "learning"])
    
    # Search for overlapping basis vectors
    search_result = tx.search(query)
    
    print(f"Search result: {search_result}")
    print(f"  Basis indices: {search_result.basis_indices}")
    print(f"  Vector IDs: {search_result.vector_ids}")
    print(f"  Query popcount: {search_result.query_popcount}")
    
    result = tx.commit()

In [ ]:
# Search with cardinality constraints
with RingTransaction(p_bits=10) as tx:
    tx.ingest(["a", "b"])  # Small batch
    tx.ingest(["c", "d", "e", "f", "g", "h", "i", "j"])  # Large batch
    
    # Query all vectors, filter by cardinality
    query = HLLSet.from_batch(["a", "b", "c", "d", "e", "f", "g", "h", "i", "j"])
    
    small = tx.search(query, max_card=5)
    large = tx.search(query, min_card=5)
    
    print(f"Small vectors (card ≤ 5): {small}")
    print(f"Large vectors (card ≥ 5): {large}")
    
    result = tx.commit()

## 10. Selective Commit

You can commit only selected basis vectors, useful for filtering relevant content.

In [20]:
with RingTransaction(p_bits=10) as tx:
    tx.ingest(["relevant", "important", "data"])
    tx.ingest(["noise", "irrelevant", "junk"])
    tx.ingest(["useful", "information"])
    
    # Only commit first and third batches (basis indices 0 and some others)
    # Get basis vectors overlapping with "relevant" content
    relevant_query = HLLSet.from_batch(["relevant", "useful", "important"])
    found = tx.search(relevant_query)
    
    # Commit only relevant bases
    result = tx.commit(basis_indices=found.basis_indices)

print(f"Selective commit: {result}")
print(f"  Selected bases: {found.basis_indices}")
print(f"  Merged cardinality: ~{result.merged_hllset.cardinality():.0f}")

Selective commit: TransactionResult(50383d13…, |M|≈6, RingDelta(rank 0→3, +3 bases, 8 tokens in 3 batches))
  Selected bases: (0, 1)
  Merged cardinality: ~6


## 11. The Token LUT

The `TokenLUT` (Lookup Table) maps (register, zeros) positions back to the original tokens that hashed there.

In [22]:
with RingTransaction(p_bits=10) as tx:
    tx.ingest(["machine", "learning", "deep", "neural"])
    result = tx.commit()

lut = result.token_lut
print(f"TokenLUT: {len(lut)} entries")
print(f"\nSample entries:")
count = 0
for (reg, zeros), entries in lut.entries.items():
    for entry in entries:
        print(f"  ({reg}, {zeros}) → '{entry.token}'")
        count += 1
        if count >= 5:
            break
    if count >= 5:
        break

TokenLUT: 4 entries

Sample entries:
  (610, 1) → 'machine'
  (382, 2) → 'learning'
  (422, 0) → 'deep'
  (1010, 1) → 'neural'


## 12. Practical Example: Document Processing Pipeline

Let's build a complete document processing pipeline using transactions.

In [23]:
# Simulate a document corpus
documents = [
    ("doc1", "machine learning is transforming artificial intelligence"),
    ("doc2", "deep neural networks achieve remarkable accuracy"),
    ("doc3", "natural language processing enables text understanding"),
    ("doc4", "computer vision algorithms detect objects in images"),
]

# Process all documents in a single transaction
with RingTransaction(p_bits=10) as tx:
    for doc_id, text in documents:
        tokens = text.lower().split()
        record = tx.ingest(tokens, label=doc_id)
        print(f"  {doc_id}: {len(tokens)} tokens, Δrank={record.rank_delta:+d}")
    
    result = tx.commit()

print(f"\nCorpus summary:")
print(f"  Total documents: {len(documents)}")
print(f"  Ring delta: {result.ring_delta}")
print(f"  Compression ratio: {result.ring_delta.compression_ratio:.2f}x")

  doc1: 6 tokens, Δrank=+1
  doc2: 6 tokens, Δrank=+1
  doc3: 6 tokens, Δrank=+1
  doc4: 7 tokens, Δrank=+1

Corpus summary:
  Total documents: 4
  Ring delta: RingDelta(rank 0→4, +4 bases, 25 tokens in 4 batches)
  Compression ratio: 6.25x


In [24]:
# The merged HLLSet represents the entire corpus vocabulary
corpus_hll = result.merged_hllset

# Check coverage of a new query
query_tokens = ["machine", "learning", "quantum", "computing"]
query_hll = HLLSet.from_batch(query_tokens)

intersection = query_hll.intersect(corpus_hll)
difference = query_hll.diff(corpus_hll)

print(f"Query analysis:")
print(f"  Query: {query_tokens}")
print(f"  In corpus: ~{intersection.cardinality():.0f}")
print(f"  Not in corpus: ~{difference.cardinality():.0f}")

Query analysis:
  Query: ['machine', 'learning', 'quantum', 'computing']
  In corpus: ~3
  Not in corpus: ~3


## Summary

In this tutorial, you learned:

1. **Transaction Lifecycle** — begin → ingest → commit/rollback
2. **IICA Enforcement** — Immutability, Idempotence, Composability, Algebraic Closure
3. **Ingest Records** — Provenance tracking with batch IDs, vector IDs, rank deltas
4. **Ring Delta** — Compression statistics and basis changes
5. **Basis Inspection** — Pre-commit introspection of the draft ring
6. **Search** — Finding relevant vectors with HLLSet queries
7. **Selective Commit** — Committing only selected basis vectors

### Key Design Points

- **Ephemeral workspace** — All changes are isolated until commit
- **Deep copy isolation** — Base ring is never mutated
- **Frozen results** — TransactionResult is completely immutable
- **Provenance chain** — Every token can be traced back

### IICA Compliance

| Principle | How Transactions Enforce It |
|-----------|----------------------------|
| Immutability | Committed results are frozen tuples |
| Idempotence | Same tokens → same bitvector → same SHA1 |
| Composability | HLLSets in, HLLSets out |
| Algebraic Closure | Ring algebra for lossless basis extraction |

### Next Steps

- **Tutorial 07**: BSS — Bell State Similarity for directed morphism testing